# 01 — Data Collection, Cleaning & Exploratory Analysis

**Yogurt Park Flavor Assortment Optimization**  
*IEOR 145 — Revenue Management | UC Berkeley*

---

## Overview

Yogurt Park, a popular frozen yogurt shop in Berkeley, CA, rotates its daily flavor lineup from a catalog of 66 flavors across 4 rotational slots (plus 2 permanently fixed: Ghirardelli Chocolate and Vanilla Classic). Poor daily selections lead to lost customers, wasted prep costs, and spoilage.

This notebook covers the full data pipeline:
1. **Data Loading** — 308 days of Instagram post data (Jan–Nov 2025)
2. **Cleaning** — date parsing, missing value removal, fixed flavor removal
3. **Flavor Analysis** — frequency distributions, descriptor extraction
4. **Categorization** — manual grouping of 66 flavors into 9 categories
5. **Feature Engineering** — category counts, descriptor flags, temporal features
6. **EDA** — what actually drives Instagram likes?
7. **Train/Test Split** — temporal 75/25 split

The output CSVs from this notebook feed directly into `02_modeling.ipynb`.


In [ ]:
!pip install statsmodels -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


## 1. Data Loading

Yogurt Park posts their daily flavor lineup on Instagram each morning. We scraped 308 posts (Jan 1 – Nov 4, 2025), recording:
- The 6 daily flavors (F1–F6)
- The number of likes on each post

Instagram likes serve as a proxy for aggregate customer preference following Berry (1994)'s framework: posts with higher engagement signal combinations that attract more interest and drive foot traffic.

> **Note:** Flavors F5 and F6 are always Ghirardelli Chocolate and Vanilla Classic. We drop them since they provide no variation for modeling.


In [ ]:
# Load raw scraped data
df = pd.read_csv('../data/raw/yogurt_park_V3.csv')

# Drop trailing empty column if present
if len(df.columns) > 8:
    df = df.drop(df.columns[8], axis=1)

# Standardize column names
df.columns = ['date', 'flavor_1', 'flavor_2', 'flavor_3',
              'flavor_4', 'flavor_5', 'flavor_6', 'instagram_likes']

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nInstagram likes stats:")
print(df['instagram_likes'].describe())


## 2. Data Cleaning

Key cleaning steps:
- **Drop missing rows** — any day with incomplete flavor or likes data
- **Parse dates** — raw dates are `M/D` strings (e.g. `11/4`); we reconstruct as `2025-MM-DD`
- **Sort chronologically** — oldest to newest
- **Add temporal features** — day of week, weekend flag, month, week of year
- **Strip fixed flavors** — drop `flavor_5` (Ghirardelli Chocolate) and `flavor_6` (Vanilla Classic)

After cleaning: **293 usable days** (Jan 2 – Nov 4, 2025).


In [ ]:
def clean_yogurt_data(df):
    """
    Clean and standardize the yogurt park dataset.
    Data range: January 1, 2025 to November 4, 2025

    - Drops flavor_5 and flavor_6 (fixed flavors: Ghirardelli Chocolate & Vanilla Classic)
    - Only keeps flavor_1 through flavor_4 (rotational flavors)
    """
    df_clean = df.copy()

    # Step 1: Drop missing values
    print(f"Before cleaning: {len(df_clean)} rows")
    print(f"Rows with missing values: {df_clean.isnull().any(axis=1).sum()}")
    df_clean = df_clean.dropna()
    print(f"After removing missing: {len(df_clean)} rows")

    # Step 2: Parse M/D date strings → 2025-MM-DD timestamps
    def parse_yogurt_date(date_str):
        try:
            month, day = map(int, date_str.split('/'))
            return pd.Timestamp(year=2025, month=month, day=day)
        except:
            return pd.NaT

    df_clean['date'] = df_clean['date'].apply(parse_yogurt_date)
    df_clean = df_clean.dropna(subset=['date'])
    print(f"After date parsing: {len(df_clean)} rows")

    # Step 3: Sort chronologically
    df_clean = df_clean.sort_values('date').reset_index(drop=True)

    # Step 4: Add temporal features
    df_clean['day_of_week']  = df_clean['date'].dt.day_name()
    df_clean['is_weekend']   = df_clean['day_of_week'].isin(['Saturday', 'Sunday'])
    df_clean['month']        = df_clean['date'].dt.month
    df_clean['month_name']   = df_clean['date'].dt.month_name()
    df_clean['week_of_year'] = df_clean['date'].dt.isocalendar().week
    df_clean['year']         = df_clean['date'].dt.year

    # Step 5: Strip whitespace from flavor names
    for col in ['flavor_1', 'flavor_2', 'flavor_3', 'flavor_4']:
        df_clean[col] = df_clean[col].str.strip()

    # Step 6: Drop fixed flavor columns
    df_clean = df_clean.drop(columns=['flavor_5', 'flavor_6'])
    print("\n✅ Dropped fixed flavor columns: flavor_5 (Ghirardelli), flavor_6 (Vanilla)")

    # Step 7: Add day index
    df_clean['day_number'] = range(1, len(df_clean) + 1)

    print(f"\nCleaning complete: {len(df_clean)} days")
    print(f"Date range: {df_clean['date'].min().date()} → {df_clean['date'].max().date()}")
    print(f"Likes: mean={df_clean['instagram_likes'].mean():.1f}, median={df_clean['instagram_likes'].median():.1f}")

    return df_clean

df_clean = clean_yogurt_data(df)
df_clean.to_csv('../data/processed/yogurt_park_clean.csv', index=False)
print("\n✓ Saved: data/processed/yogurt_park_clean.csv")


## 3. Flavor Analysis

With fixed flavors removed, we analyze the 4 rotational slots across all 293 days.

Key questions:
- How many unique flavors appear in the rotation?
- Which flavors appear most frequently?
- What descriptors (Sugar Free, Low Fat, Sorbet) are most common?


In [ ]:
rotational_cols = ['flavor_1', 'flavor_2', 'flavor_3', 'flavor_4']
n_days = len(df_clean)

# Flatten all rotational flavor appearances
rotational_flavors_list = []
for col in rotational_cols:
    rotational_flavors_list.extend(df_clean[col].tolist())

rotational_counts = pd.Series(rotational_flavors_list).value_counts()

print(f"Total unique rotational flavors: {len(rotational_counts)}")
print(f"Total rotational slots: {len(rotational_flavors_list)} (4/day × {n_days} days)")
print(f"Average appearances per flavor: {len(rotational_flavors_list) / len(rotational_counts):.1f} times")

print(f"\nTop 20 Most Frequent Rotational Flavors:")
print("-"*70)
for i, (flavor, count) in enumerate(rotational_counts.head(20).items(), 1):
    pct = (count / n_days) * 100
    bar = '█' * int(pct / 5)
    print(f"{i:2d}. {flavor:50s} {count:3d} days ({pct:5.1f}%) {bar}")

# Build flavor_info table
flavor_info = pd.DataFrame({
    'flavor': rotational_counts.index,
    'days_appeared': rotational_counts.values,
    'appearance_rate': rotational_counts.values / n_days,
    'times_per_week': (rotational_counts.values / n_days) * 7
})
flavor_info['base_flavor'] = flavor_info['flavor'].str.replace(r'\s*\([^)]*\)', '', regex=True).str.strip()

def get_descriptor(name):
    if '(SF)' in name:      return 'Sugar Free'
    if '(Low Fat)' in name: return 'Low Fat'
    if '(Sorbet)' in name:  return 'Sorbet'
    if '(Gelato)' in name:  return 'Gelato'
    return 'Regular'

flavor_info['descriptor'] = flavor_info['flavor'].apply(get_descriptor)

def categorize_frequency(rate):
    if rate >= 0.5:  return 'High Frequency'
    if rate >= 0.25: return 'Medium Frequency'
    if rate >= 0.10: return 'Low Frequency'
    return 'Rare'

flavor_info['frequency_category'] = flavor_info['appearance_rate'].apply(categorize_frequency)
flavor_info = flavor_info.sort_values('days_appeared', ascending=False).reset_index(drop=True)

print(f"\nFlavor Descriptors (rotational only):")
print(flavor_info['descriptor'].value_counts())


## 4. Flavor Categorization

With 66 unique flavors, estimating individual flavor effects would require far more data than we have (~18 appearances per flavor on average). Instead, we manually group flavors into **9 broad categories** based on taste profile:

| Category | Count | Examples |
|---|---|---|
| Chocolate | 11 | Cookies & Cream, Thin Mint Cookie, Dark Chocolate |
| Vanilla | 7 | White Cake Batter, Birthday Cake, French Vanilla |
| Fruit | 19 | Mango, Fresh Blackberry, Red Raspberry |
| Caramel/Dulce | 11 | Dulce de Leche, Salted Caramel, Caramel Custard |
| Nut | 17 | Pistachio Cream, Hazelnut Amaretto, Pecans & Pralines |
| Tart | 6 | Original Tart, Greek Yogurt, Pomegranate Tart |
| Coffee/Matcha | 9 | Costa Rican Cappuccino, Matcha Green Tea, Kahlua |
| Sorbet | 5 | Passion Orange Guava, Mango Sorbet, Lemon Sorbet |
| Specialty | 27 | Creme Brulee, NY Cheesecake, Strawberry Shortcake |

This grouping follows Berry, Levinsohn & Pakes (1995)'s attribute-based approach: customers generally prefer flavor *profiles* (e.g. "something fruity") rather than specific SKUs. Category-level features also produce actionable recommendations ("feature more fruit flavors this weekend") that store managers can act on.


In [ ]:
flavor_categories = {
    'Chocolate': [
        'Cookies & Cream', 'Peanut Butter Chocolate', 'Dark Chocolate',
        'Chocolate', 'Nutella', 'Ferrero Rocher', 'Thin Mint Cookie',
        'Black Forest Chocolate', 'Mounds Bar', 'Creamy Milk Chocolate', 'Belgian Chocolate'
    ],
    'Vanilla': [
        'Vanilla Bean', 'French Vanilla', 'Cake Batter', 'Birthday Cake',
        'White Cake Batter', 'Yellow Cake Batter', 'Yellow Cake Batter (SF)'
    ],
    'Fruit': [
        'Fresh Blackberry', 'Strawberry', 'Mango', 'Blueberry', 'Peach',
        'Raspberry', 'Watermelon', 'Passion Fruit', 'Mixed Berry', 'Pomegranate',
        'Red Raspberry', 'Fresh Strawberry', 'Hawaiian Coconut', 'Maine Blueberry',
        'Wild Mountain Blueberries', 'Georgia Peach', 'Wild Berry (Tart)',
        'Wild Mountain Berries', 'Blueberry (Tart)'
    ],
    'Caramel/Dulce': [
        'Dulce de Leche (Low Fat)', 'Dulce de Leche', 'Salted Caramel', 'Caramel',
        'Caramel Macadamia Cappuccino', 'Caramel Custard (SF)',
        'Caramel Macadamia Cappuccino (SF)', 'Butterscotch Parfait (SF)',
        'Salted Caramel Pretzel (Low Fat)', 'Salted Caramel Macadamia', 'Caramel Custard'
    ],
    'Nut': [
        'Hazelnut Amaretto (SF)', 'Hazelnut Amaretto', 'Pistachio Cream (SF)',
        'Pistachio', 'Peanut Butter', 'Almond', 'Butter Pecan (SF)',
        'Peanut Butter (Low Fat)', 'Hazelnut (SF)', 'Maple Walnut (SF)',
        'Pistachio Cream', 'Pecan and Pralines (SF)', 'Butter Pecan', 'Hazelnut',
        'Pistachio (SF)', 'Pecans & Pralines (SF)', 'Salted Caramel Macadamia (Low Fat)'
    ],
    'Tart': [
        'Original Tart', 'Greek Yogurt', 'Pomegranate Tart', 'Tart',
        'Pomegranate Raspberry (Tart)', 'Blueberry (Tart)'
    ],
    'Coffee/Matcha': [
        'Coffee', 'Espresso', 'Matcha', 'Matcha Green Tea', 'Cappuccino',
        'Costa Rican Cappuccino', 'Vanilla Cappuccino (SF)', 'Vanilla Cappuccino', 'Kahlua (SF)'
    ],
    'Sorbet': [
        'Passion Orange Guava (Sorbet)', 'Mango (Sorbet)', 'Raspberry (Sorbet)',
        'Lemon (Sorbet)', 'Watermelon (Sorbet)'
    ],
    'Specialty': [
        'Cheesecake', 'Pumpkin', 'Eggnog', 'Red Velvet', 'Mint Chocolate Chip',
        'Cookie Dough', 'Creme Brulee', 'NY Cheesecake', 'Irish Mint', 'Pumpkin Pie',
        'Chocolate Covered Banana', 'Rocky Road', 'English Toffee (SF)',
        'White Chocolate Macadamia (SF)', 'NY Cheesecake (SF)', 'White Chocolate Macadamia',
        'Strawberry Shortcake', 'Salted Caramel Macadamia', 'Chocolate Covered Strawberries',
        'Raspberry Chocolate Truffle', 'Orange Creamsicle', 'Apple Pie', 'Banana Chocolate',
        'Caramel Pecan Toffee', 'Caramel Pecan Toffee (SF)', 'Creme Brulee (SF)',
        'Butterscotch Parfait (SF)'
    ]
}

# Build reverse lookup
flavor_to_category = {flavor: cat for cat, flavors in flavor_categories.items() for flavor in flavors}

# Apply to flavor_info
flavor_info['category'] = flavor_info['flavor'].map(flavor_to_category)

unmapped = flavor_info[flavor_info['category'].isna()]
if len(unmapped) > 0:
    print(f"⚠️  {len(unmapped)} unmapped flavors (assigned to 'Other'):")
    for _, row in unmapped.iterrows():
        print(f"   • {row['flavor']} ({row['days_appeared']} days)")
    flavor_info.loc[flavor_info['category'].isna(), 'category'] = 'Other'
else:
    print("✓ All flavors successfully categorized!")

# Category distribution
print("\nCategory Distribution:")
print("="*60)
category_summary = flavor_info.groupby('category').agg(
    n_flavors=('flavor', 'count'),
    total_appearances=('days_appeared', 'sum'),
    avg_appearance_rate=('appearance_rate', 'mean')
).round(3).sort_values('total_appearances', ascending=False)
print(category_summary)

flavor_info.to_csv('../data/processed/flavor_summary.csv', index=False)
print("\n✓ Saved: data/processed/flavor_summary.csv")


## 5. Data Quality Checks

Before modeling, we verify:
1. No duplicate flavors within a single day
2. Instagram likes distribution (outliers, variance)
3. Temporal coverage (gaps in daily data)
4. Balanced day-of-week representation
5. Month-by-month coverage


In [ ]:
print("="*70)
print("DATA QUALITY CHECKS")
print("="*70)

# 1. Duplicate flavors within days
duplicate_days = 0
for idx, row in df_clean.iterrows():
    flavors = [row[f'flavor_{i}'] for i in range(1, 5)]
    if len(flavors) != len(set(flavors)):
        duplicate_days += 1

if duplicate_days == 0:
    print("\n1. ✓ No duplicate flavors within any day")
else:
    print(f"\n1. ⚠ {duplicate_days} days with duplicate flavors")

# 2. Likes distribution
print("\n2. Instagram likes distribution:")
print(f"   Mean:    {df_clean['instagram_likes'].mean():.2f}")
print(f"   Median:  {df_clean['instagram_likes'].median():.2f}")
print(f"   Std:     {df_clean['instagram_likes'].std():.2f}")
print(f"   Min/Max: {df_clean['instagram_likes'].min():.0f} / {df_clean['instagram_likes'].max():.0f}")

mean_likes, std_likes = df_clean['instagram_likes'].mean(), df_clean['instagram_likes'].std()
outliers = df_clean[np.abs(df_clean['instagram_likes'] - mean_likes) > 2 * std_likes]
print(f"   Outliers (>2σ): {len(outliers)}")
for _, row in outliers.iterrows():
    flavors = [row[f'flavor_{i}'] for i in range(1, 5)]
    print(f"     {row['date'].date()}: {row['instagram_likes']:.0f} likes | {', '.join(flavors)}")

# 3. Temporal coverage
date_gaps = df_clean['date'].diff().dt.days
large_gaps = date_gaps[date_gaps > 1].dropna()
print(f"\n3. Temporal gaps (>1 day): {len(large_gaps)}")
for idx in large_gaps.index:
    gap = (df_clean.loc[idx, 'date'] - df_clean.loc[idx-1, 'date']).days
    print(f"   Gap of {gap} days: {df_clean.loc[idx-1, 'date'].date()} → {df_clean.loc[idx, 'date'].date()}")

# 4. Day-of-week distribution
print("\n4. Day of week distribution:")
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = df_clean['day_of_week'].value_counts()
for day in day_order:
    if day in dow_counts.index:
        count = dow_counts[day]
        bar = '█' * int(count / len(df_clean) * 50)
        print(f"   {day:10s}: {count:3d} ({count/len(df_clean)*100:5.1f}%) {bar}")

print(f"\n   Weekend: {df_clean['is_weekend'].sum()} days | Weekday: {(~df_clean['is_weekend']).sum()} days")

# 5. Month distribution
print("\n5. Month distribution:")
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November']
for month in month_order:
    count = (df_clean['month_name'] == month).sum()
    if count > 0:
        bar = '█' * int(count / 5)
        print(f"   {month:10s}: {count:3d} days {bar}")


## 6. Feature Engineering

We transform each day's 4-flavor lineup into quantitative features for modeling:

**Category counts** (`n_chocolate`, `n_fruit`, etc.)  
For each of the 9 flavor categories, how many of the day's 4 flavors belong to it? This lets the model learn that, e.g., a day with 2 fruit flavors performs differently than one with 1.

**Descriptor flags** (`n_sugar_free`, `n_low_fat`, `n_sorbet`)  
How many flavors carry health-modifier labels? These test whether dietary-specific options broaden customer appeal.

**Diversity metric** (`n_categories`)  
How many distinct flavor categories appear that day? Tests the "sweet spot" hypothesis — enough variety to be interesting, not so much that no flavor stands out.


In [ ]:
def add_assortment_features(df_clean, flavor_info):
    """
    Create day-level assortment features:
    - Category counts: how many flavors per category
    - Diversity: number of distinct categories
    - Descriptor counts: sugar free, low fat, sorbet
    """
    df_features = df_clean.copy()
    flavor_to_cat = dict(zip(flavor_info['flavor'], flavor_info['category']))
    flavor_cols = ['flavor_1', 'flavor_2', 'flavor_3', 'flavor_4']
    categories = flavor_info['category'].unique()

    # Initialize count columns
    for cat in categories:
        df_features[f'n_{cat.lower().replace("/", "_")}'] = 0
    df_features['n_categories'] = 0
    df_features['n_sugar_free'] = 0
    df_features['n_low_fat']    = 0
    df_features['n_sorbet']     = 0

    for idx, row in df_features.iterrows():
        cats_present = []
        for col in flavor_cols:
            flavor = row[col]
            cat = flavor_to_cat.get(flavor, 'Other')
            col_name = f'n_{cat.lower().replace("/", "_")}'
            if col_name in df_features.columns:
                df_features.loc[idx, col_name] += 1
            if cat not in cats_present:
                cats_present.append(cat)
            if '(SF)' in flavor:
                df_features.loc[idx, 'n_sugar_free'] += 1
            if '(Low Fat)' in flavor:
                df_features.loc[idx, 'n_low_fat'] += 1
            if '(Sorbet)' in flavor:
                df_features.loc[idx, 'n_sorbet'] += 1
        df_features.loc[idx, 'n_categories'] = len(cats_present)

    return df_features

df_features = add_assortment_features(df_clean, flavor_info)

print("Feature columns created:")
feature_cols = [col for col in df_features.columns if col.startswith('n_')]
print(feature_cols)
print(f"\nSample (first 5 rows):")
print(df_features[['date', 'instagram_likes'] + feature_cols].head())

df_features.to_csv('../data/processed/yogurt_park_features.csv', index=False)
print("\n✓ Saved: data/processed/yogurt_park_features.csv")


## 7. Exploratory Analysis

Before modeling, we examine what the raw data suggests about engagement drivers.

Key questions:
- Does weekend vs. weekday matter?
- Does category diversity help?
- Which days perform best/worst, and why?
- How correlated are individual features with likes?

> **Spoiler:** The raw correlations are weak (max ~0.11), which motivates the MNL framework — context-dependent interactions matter more than simple linear effects.


In [ ]:
print("="*70)
print("EXPLORATORY ANALYSIS: WHAT DRIVES INSTAGRAM LIKES?")
print("="*70)

# 1. Weekend effect
print("\n1. Weekend vs Weekday Performance:")
weekend_stats = df_features.groupby('is_weekend')['instagram_likes'].agg(['mean', 'median', 'count'])
weekend_stats.index = ['Weekday', 'Weekend']
print(weekend_stats)

# 2. Category diversity effect
print("\n2. Likes by Number of Distinct Categories:")
cat_analysis = df_features.groupby('n_categories')['instagram_likes'].agg(['mean', 'median', 'count'])
print(cat_analysis)

# 3. Top 10 performing days
print("\n3. Top 10 Performing Days:")
top_days = df_features.nlargest(10, 'instagram_likes')[
    ['date', 'day_of_week', 'flavor_1', 'flavor_2', 'flavor_3', 'flavor_4', 'instagram_likes']
]
print(top_days.to_string(index=False))

# 4. Bottom 10 performing days
print("\n4. Bottom 10 Performing Days:")
bottom_days = df_features.nsmallest(10, 'instagram_likes')[
    ['date', 'day_of_week', 'flavor_1', 'flavor_2', 'flavor_3', 'flavor_4', 'instagram_likes']
]
print(bottom_days.to_string(index=False))

# 5. Correlation with likes
print("\n5. Feature Correlations with Instagram Likes:")
corr_cols = ['instagram_likes', 'n_categories', 'is_weekend',
             'n_sugar_free', 'n_low_fat', 'n_sorbet'] +             [col for col in df_features.columns
             if col.startswith('n_') and col not in ['n_categories', 'n_sugar_free', 'n_low_fat', 'n_sorbet']]
corr_with_likes = df_features[corr_cols].corr()['instagram_likes'].sort_values(ascending=False)
print(corr_with_likes.to_string())


## 8. Train / Test Split

We use a **temporal split** (not random) to respect the time-series structure of the data and simulate a realistic deployment scenario: use past data to predict future assortments.

- **Training set:** Jan 2 – Aug 18, 2025 (219 days, 75%)
- **Test set:** Aug 19 – Nov 4, 2025 (74 days, 25%)

> **Important caveat:** The test set averages 13.6 likes vs. 10.5 in training — a 30% increase. This likely reflects a growing Instagram following or the start of the UC Berkeley fall semester driving more engagement. This distribution shift is a realistic modeling challenge and is noted in the final report.


In [ ]:
print("="*70)
print("TRAIN-TEST SPLIT")
print("="*70)

# Temporal split: 75% train, 25% test
split_date = df_features['date'].quantile(0.75)

train_df = df_features[df_features['date'] < split_date].copy()
test_df  = df_features[df_features['date'] >= split_date].copy()

print(f"\nTraining set: {len(train_df)} days")
print(f"  {train_df['date'].min().date()} → {train_df['date'].max().date()}")
print(f"  Likes: mean={train_df['instagram_likes'].mean():.1f}, std={train_df['instagram_likes'].std():.1f}")

print(f"\nTest set: {len(test_df)} days")
print(f"  {test_df['date'].min().date()} → {test_df['date'].max().date()}")
print(f"  Likes: mean={test_df['instagram_likes'].mean():.1f}, std={test_df['instagram_likes'].std():.1f}")

print(f"\n⚠️  Note: Test set mean ({test_df['instagram_likes'].mean():.1f}) is ~30% higher than train ({train_df['instagram_likes'].mean():.1f}).")
print("   Likely due to growing Instagram following or fall semester return to campus.")
print("   Models trained on the earlier period may systematically underpredict test outcomes.")

train_df.to_csv('../data/processed/train_data.csv', index=False)
test_df.to_csv('../data/processed/test_data.csv',  index=False)
print("\n✓ Saved: data/processed/train_data.csv")
print("✓ Saved: data/processed/test_data.csv")
